In [ ]:
import os
os.makedirs("Figs", exist_ok=True)  # figures are written to ./Figs

import json
import math
from pathlib import Path

import numpy as np

# ------------------------------------------------------------------ #
# Same ground-truth parsing as eval_metrics.py: None/NaN handling and the
# no-N/H-for-proline and no-CB-for-glycine exclusions, identical to training/validation.
# ------------------------------------------------------------------ #
def parse_shift_list(lst, sequence, atom):
    vals, mask = [], []
    for i, x in enumerate(lst):
        residue = sequence[i]
        bad = False
        if x is None:
            bad = True
        elif isinstance(x, str):
            xs = x.strip().lower()
            if xs in {"none", "null", "nan", "inf", "+inf", "-inf"}:
                bad = True
            else:
                try:
                    x = float(x)
                except Exception:
                    bad = True
        if not bad and isinstance(x, (int, float)):
            if not math.isfinite(float(x)):
                bad = True
        if atom in ("N", "H") and residue == "P":
            bad = True
        if atom == "CB" and residue == "G":
            bad = True
        if bad:
            vals.append(0.0)
            mask.append(False)
        else:
            vals.append(float(x))
            mask.append(True)
    return vals, mask


def load_ground_truth(gt_path: Path, atom: str):
    """entry_ID (without the bmr prefix) -> (shift list, mask list)."""
    with open(gt_path) as f:
        rbmrb = json.load(f)
    out = {}
    for ref_id, shift_lst in rbmrb[atom].items():
        if all(x is None or (isinstance(x, str) and x.lower() in ("none", "null"))
               for x in shift_lst):
            continue
        sequence = rbmrb["sequence"][ref_id]
        shift, mask = parse_shift_list(shift_lst, sequence, atom=atom)
        entry = rbmrb["entry_ID"][ref_id].lower().removeprefix("bmr")
        out[entry] = (shift, mask)
    return out


def load_pred(path):
    """In predictions_N_seed_ensemble_*.json, "pred" is the per-seed [n_seeds, n_res] matrix;
    evaluation uses the precomputed per-residue seed mean "mean" (with "mask")."""
    with open(path) as f:
        d = json.load(f)
    return d["mean"], d["mask"]


def build_per_protein(pred_path, gt_path, atom):
    """entry -> (pred_vals, gt_vals), aligned exactly as in eval_metrics.py:
    only the prediction's own mask intersected with the GT None/Pro-N-H/Gly-CB mask.
    The tag/flag/extra masks shipped with each test set (N_mask, *_flag, ...) were already folded
    into the prediction mask when predictions were generated, so they are not applied again."""
    pred_dict, mask_dict = load_pred(pred_path)
    gt = load_ground_truth(Path(gt_path), atom=atom)

    per_protein = {}
    skipped = []
    for entry, (true_shift, true_mask) in gt.items():
        if entry not in pred_dict:
            skipped.append(entry)
            continue
        pred_shift = pred_dict[entry]
        pred_mask = mask_dict[entry]
        n = min(len(true_shift), len(true_mask), len(pred_shift), len(pred_mask))
        combined_mask = np.array(true_mask[:n], dtype=bool) & np.array(pred_mask[:n], dtype=bool)
        if not combined_mask.any():
            skipped.append(entry)
            continue
        per_protein[entry] = {
            "pred": np.asarray(pred_shift[:n], dtype=float)[combined_mask],
            "gt":   np.asarray(true_shift[:n], dtype=float)[combined_mask],
        }
    return per_protein, skipped


def rmse(pred, gt):
    return float(np.sqrt(np.mean((pred - gt) ** 2)))


def bootstrap_rmse(per_protein, B=2000, seed=42):
    """Protein-level bootstrap; returns (overall_rmse, SE)."""
    proteins = list(per_protein.keys())
    n = len(proteins)
    all_pred = np.concatenate([per_protein[e]["pred"] for e in proteins])
    all_gt   = np.concatenate([per_protein[e]["gt"] for e in proteins])
    overall = rmse(all_pred, all_gt)

    rng = np.random.default_rng(seed)
    boot = []
    for _ in range(B):
        sampled = rng.choice(proteins, size=n, replace=True)
        p = np.concatenate([per_protein[e]["pred"] for e in sampled])
        t = np.concatenate([per_protein[e]["gt"] for e in sampled])
        boot.append(rmse(p, t))
    se = float(np.std(np.array(boot)))
    return overall, se

In [ ]:
RESULTS_DIR = Path("../../data/analysis/Fig2/results")
ATOM = "N"

# Evaluate every model (prediction file) x test set (GT file) combination
models = ["panav", "lacs", "noref"]
model_pred_path = {m: RESULTS_DIR / f"predictions_N_seed_ensemble_10seed_{m}.json" for m in models}

testsets = ["panav", "lacs", "noref"]
testset_gt_path = {t: RESULTS_DIR / f"test_{t}.json" for t in testsets}

# results_by_testset[testset][model] = (RMSE, bootstrap SE)
results_by_testset = {t: {} for t in testsets}

for t in testsets:
    for m in models:
        per_protein, skipped = build_per_protein(model_pred_path[m], testset_gt_path[t], ATOM)
        overall, se = bootstrap_rmse(per_protein)
        results_by_testset[t][m] = (overall, se)
        print(f"test={t:6s} model={m:6s}  n_proteins={len(per_protein)} (skipped {len(skipped)})  "
              f"RMSE={overall:.4f}  SE={se:.4f}")

In [ ]:
import matplotlib.pyplot as plt

def set_style(base=12, family="Arial"):
    plt.rcParams.update({
        "font.family": family,
        "font.size": base - 1,
        "axes.titlesize": base + 2,
        "axes.labelsize": base,
        "xtick.labelsize": base - 3,
        "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2,
        "svg.fonttype": "none",
    })

set_style(base=18)

testset_labels = {"lacs": "rBMRB-LACS\nTest Set", "panav": "rBMRB-PANAV\nTest Set", "noref": "rBMRB-BASELINE\nTest Set"}
model_labels    = {"lacs": "SLAE-CS-LACS model", "panav": "SLAE-CS-PANAV model", "noref": "SLAE-CS-BASELINE model"}
model_colors    = {"noref": "#999999", "panav": "#61AFEF", "lacs": "#E06C75"}

plot_testsets = testsets
plot_models   = models

x        = np.arange(len(plot_testsets))
n_models = len(plot_models)
width    = 0.8 / n_models

fig, ax = plt.subplots(figsize=(8, 5))

all_means = []
for i, m in enumerate(plot_models):
    means = [results_by_testset[t][m][0] for t in plot_testsets]
    ses   = [results_by_testset[t][m][1] for t in plot_testsets]
    all_means += means
    offset = (i - (n_models - 1) / 2) * width
    bars = ax.bar(x + offset, means, width, yerr=ses, capsize=4,
                  color=model_colors[m], edgecolor="white", linewidth=0.6,
                  label=model_labels[m],
                  error_kw=dict(elinewidth=1.2, ecolor="gray", capthick=1.2))
    for bar, mean in zip(bars, means):
        print(f"{mean:.3f}")
        ax.text(bar.get_x() + bar.get_width() / 2, mean + 0.07,
                f"{mean:.3f}", ha="center", va="bottom")

ax.set_xticks(x)
ax.set_xticklabels([testset_labels[t] for t in plot_testsets])
ax.set_ylabel("RMSE (ppm, mean ± bootstrap SE)")
ax.set_ylim(max(0, min(all_means) - 0.4), max(all_means) + 0.4)
ax.legend(frameon=False, ncol=1, loc="upper left")
# ax.grid(axis="y", alpha=0.3, linestyle="--")
plt.tight_layout()
plt.savefig("Figs/rmse_comparison_seedensemble_acrossmodel.png", dpi=300, bbox_inches="tight")
plt.savefig("Figs/rmse_comparison_seedensemble_acrossmodel.svg", format='svg', bbox_inches="tight")
plt.show()